# 04 · Cells high in S1 / S7 / S12 / S15 — the groups for the multiome accessibility analysis

Four activity programmes of the model are followed into chromatin accessibility on the multiome
nuclei. This notebook picks, in every cell of both arms, the cells that run each programme and
gives every cell **one** label:

| factor | programme |
|---|---|
| S1 | TGFb (TGFb / hypoxia / mechano-activation) |
| S7 | Inflammatory |
| S12 | Inflammatory stress (HMOX1 / HAS1) |
| S15 | Mature adipose |

**The rule.** A cell's usage of the 16 factors is a composition summing to 1. For each programme
its share is z-scored across all cells of the arm (donors pooled; the two arms are standardised
separately because scRNA-seq and snRNA-seq have different baselines). A cell *qualifies* for a
programme when `z ≥ MIN_Z` **and** its raw share `≥ MIN_SHARE` — at least 10 % of the cell's
usage, so a call means the cell genuinely runs the programme rather than merely sitting above the
average for it. **Among the programmes a cell qualifies for, the highest z wins**, so every cell
carries exactly one label; a cell qualifying for none is `unassigned`, and most cells are — these
are four specific activities, not a partition of fibroblast biology.

Thresholds: **`MIN_Z = 0.75`, `MIN_SHARE = 0.10`** (the setting used for the accessibility
analysis; `MIN_Z = 1.0` was the initial default and is kept in the sensitivity table).

The identity of the cells is not used anywhere in the call; the harmonised annotation is only
consulted at the end as a sanity check.

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import hdf5plugin
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

sys.path.insert(0, '/nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/notebooks/revisions/final')
import fb_common as C

C.set_style()
RES, PLT = C.out_dirs('04_programme_calls')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

light = ad.read_h5ad(C.light_path())
H = pd.read_parquet(f'{C.INMF_DIR}/H_{C.TAG}.parquet')
H = H.loc[H.index.intersection(light.obs_names)]
light = light[H.index].copy()
obs = light.obs
usage = C.usage_shares(H)
arm = obs['arm'].astype(str)
donor = obs['donor_comb'].astype(str)
harm = pd.read_csv(C.HARMONIZED_OBS, index_col=0, usecols=lambda c: c in ('', 'Unnamed: 0', C.HARMONIZED_LABEL))
population = harm[C.HARMONIZED_LABEL].reindex(obs.index).astype(str)

PROGRAMMES = C.PROGRAMMES                 # {'TGFb': 'S1', 'Inflammatory': 'S7', 'Inflammatory_stress': 'S12', 'Adipose_mature': 'S15'}
ORDER = list(PROGRAMMES) + ['unassigned']
MIN_Z, MIN_SHARE = 0.75, 0.10
MIN_PER_DONOR = 50                        # a donor contributes to a pseudobulk contrast only above this
print(f"{int((arm == 'cells').sum()):,} cells | {int((arm == 'nuclei').sum()):,} nuclei")

70,258 cells | 14,858 nuclei


## 1 · Score: share and within-arm z per programme

In [2]:
share = pd.DataFrame({name: usage[f] for name, f in PROGRAMMES.items()})
z = C.z_within(share, arm)
summary = pd.DataFrame({'factor': pd.Series(PROGRAMMES), 'name': {n: C.FACTOR_NAMES[f] for n, f in PROGRAMMES.items()},
                        'mean_share_cells': share[(arm == 'cells').values].mean().round(4),
                        'mean_share_nuclei': share[(arm == 'nuclei').values].mean().round(4)})
display(summary)

,factor,name,mean_share_cells,mean_share_nuclei
TGFb,S1,TGFb / hypoxia / mechano,0.0284,0.0466
Inflammatory,S7,inflammatory,0.1252,0.0464
Inflammatory_stress,S12,inflammatory stress,0.0591,0.0344
Adipose_mature,S15,mature adipose,0.0219,0.0294


## 2 · Assign: qualify on both thresholds, then the highest z wins

In [3]:
def assign(min_z, min_share):
    qualifies = (z >= min_z) & (share >= min_share)
    winner = z.where(qualifies, -np.inf).idxmax(axis=1).where(qualifies.any(axis=1), 'unassigned')
    calls = pd.DataFrame({'programme': winner,
                          'programme_z': z.where(qualifies).max(axis=1).round(3),
                          'n_qualified': qualifies.sum(axis=1),
                          'arm': arm.values, 'donor': donor.values, 'population': population.values}, index=usage.index)
    for name in PROGRAMMES:
        calls[f'z_{name}'] = z[name].round(3)
        calls[f'share_{name}'] = share[name].round(4)
    return calls


calls = assign(MIN_Z, MIN_SHARE)
t = pd.DataFrame({a: calls.loc[calls['arm'] == a, 'programme'].value_counts().reindex(ORDER).fillna(0).astype(int) for a in C.ARMS})
t['pct_cells'] = (t['cells'] / t['cells'].sum() * 100).round(1)
t['pct_nuclei'] = (t['nuclei'] / t['nuclei'].sum() * 100).round(1)
t.to_csv(f'{RES}/calls_per_arm.csv')
print(f'MIN_Z = {MIN_Z}, MIN_SHARE = {MIN_SHARE}')
display(t)

for a in C.ARMS:
    am = (calls['arm'] == a).values
    tied = am & (calls['n_qualified'] >= 2).values
    print(f'{a}: {int(tied.sum()):,} cells qualified for 2+ programmes; the argmax gave them to '
          + ', '.join(f'{k} {v:,}' for k, v in calls.loc[tied, 'programme'].value_counts().items()))

MIN_Z = 0.75, MIN_SHARE = 0.1


,cells,nuclei,pct_cells,pct_nuclei
programme,,,,
TGFb,3572,935,5.1,6.3
Inflammatory,11251,1810,16.0,12.2
Inflammatory_stress,5603,1547,8.0,10.4
Adipose_mature,3430,1198,4.9,8.1
unassigned,46402,9368,66.0,63.1


cells: 6,223 cells qualified for 2+ programmes; the argmax gave them to Inflammatory 2,307, TGFb 1,856, Inflammatory_stress 1,844, Adipose_mature 216
nuclei: 1,365 cells qualified for 2+ programmes; the argmax gave them to TGFb 470, Inflammatory 404, Inflammatory_stress 399, Adipose_mature 92


## 3 · Nuclei per programme per donor

The table that decides what a pseudobulk contrast on the multiome arm can do. Every nucleus is in
exactly one column, so rows sum to each donor's total. Because of the argmax a donor whose nuclei
run several programmes loses its weaker ones to its stronger ones (TGFb in A70 is the clearest
case), so the standalone sensitivity grid below is read *before* the argmax and the donor support
is read *after* it.

In [4]:
nuc = calls[calls['arm'] == 'nuclei']
per_donor = pd.crosstab(nuc['donor'], nuc['programme']).reindex(columns=ORDER).fillna(0).astype(int)
per_donor = per_donor.loc[per_donor.sum(axis=1).sort_values(ascending=False).index]
support = pd.DataFrame({'n_nuclei': per_donor.sum(), 'n_donors': (per_donor > 0).sum(),
                        f'n_donors_ge{MIN_PER_DONOR}': (per_donor >= MIN_PER_DONOR).sum()})
per_donor.to_csv(f'{RES}/nuclei_per_programme_per_donor.csv')
support.to_csv(f'{RES}/donor_support_nuclei.csv')
display(per_donor)
display(support)

fig, ax = plt.subplots(figsize=C.figsize_mm('single', 50), layout='constrained')
im = C.heatmap(ax, per_donor.drop(columns='unassigned'), 'Blues', 0, float(per_donor.drop(columns='unassigned').values.max()),
               xrot=30, annot=per_donor.drop(columns='unassigned').map(lambda v: f'{int(v):,}'))
for i in range(per_donor.shape[0]):
    for j, p in enumerate(list(PROGRAMMES)):
        if per_donor.iloc[i][p] >= MIN_PER_DONOR:
            ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, edgecolor=C.COLORS['grass'], lw=1.2))
ax.set_title(f'nuclei per programme per donor (box = ≥ {MIN_PER_DONOR})'); ax.set_ylabel('donor')
C.save(fig, f'{PLT}/nuclei_per_programme_per_donor.pdf')
plt.show()

programme,TGFb,Inflammatory,Inflammatory_stress,Adipose_mature,unassigned
donor,,,,,
A70,44,913,829,1163,6982
A67,880,707,331,3,648
A72,4,111,382,13,1202
A56,7,79,4,14,358
Z6,0,0,0,2,75
U09,0,0,0,0,30
A66,0,0,1,1,26
Z17,0,0,0,2,26
U11,0,0,0,0,15


,n_nuclei,n_donors,n_donors_ge50
programme,,,
TGFb,935,4,1
Inflammatory,1810,4,4
Inflammatory_stress,1547,5,3
Adipose_mature,1198,7,1
unassigned,9368,11,5


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/04_programme_calls/nuclei_per_programme_per_donor.pdf   89 x 50 mm


In [5]:
# threshold sensitivity: standalone qualification (before the argmax), nuclei arm
m = (arm == 'nuclei').values
rows = []
for mz in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]:
    for ms in [0.05, 0.10, 0.15]:
        for name in PROGRAMMES:
            sel = (z.loc[m, name] >= mz) & (share.loc[m, name] >= ms)
            vc = donor[m][sel.values].value_counts()
            rows.append({'programme': name, 'MIN_Z': mz, 'MIN_SHARE': ms, 'n_nuclei': int(sel.sum()),
                         f'donors_ge{MIN_PER_DONOR}': int((vc >= MIN_PER_DONOR).sum()),
                         'top_donors': '; '.join(f'{k}:{v}' for k, v in vc.head(4).items())})
grid = pd.DataFrame(rows)
grid.to_csv(f'{RES}/threshold_sensitivity_nuclei.csv', index=False)
print('nuclei arm, standalone qualification per threshold pair:')
for name in PROGRAMMES:
    print(f'-- {name} --')
    print(grid[grid.programme == name].drop(columns='programme').to_string(index=False), '\n')

nuclei arm, standalone qualification per threshold pair:
-- TGFb --
 MIN_Z  MIN_SHARE  n_nuclei  donors_ge50                          top_donors
  0.00       0.05      2667            4 A67:1601; A70:818; A56:127; A72:115
  0.00       0.10      1792            2   A67:1421; A70:310; A56:46; A72:14
  0.00       0.15      1387            2    A67:1245; A70:123; A56:17; A72:2
  0.25       0.05      2121            3   A67:1512; A70:499; A56:72; A72:36
  0.25       0.10      1792            2   A67:1421; A70:310; A56:46; A72:14
  0.25       0.15      1387            2    A67:1245; A70:123; A56:17; A72:2
  0.50       0.05      1735            2   A67:1400; A70:281; A56:40; A72:14
  0.50       0.10      1735            2   A67:1400; A70:281; A56:40; A72:14
  0.50       0.15      1387            2    A67:1245; A70:123; A56:17; A72:2
  0.75       0.05      1489            2    A67:1301; A70:162; A56:22; A72:4
  0.75       0.10      1489            2    A67:1301; A70:162; A56:22; A72:4
  0.75  

## 4 · The calls on the UMAP

In [6]:
xy = np.asarray(light.obsm['X_umap'])
fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('double', 78), layout='constrained')
for ax, a in zip(axes, C.ARMS):
    am = (arm == a).values
    lab = calls['programme'].values.copy()
    C.umap_categorical(xy, lab, C.PROG_COLORS, order=list(PROGRAMMES), ax=ax, mask=am, point_size=0.5 if a == 'cells' else 0.9,
                       title=f'{C.ARM_NAMES[a]} — {int((am & (lab != "unassigned")).sum()):,} assigned, {int((am & (lab == "unassigned")).sum()):,} unassigned')
C.save(fig, f'{PLT}/programme_calls_umap.pdf')
plt.show()

fig, axes = plt.subplots(2, len(PROGRAMMES), figsize=C.figsize_mm('double', 95), layout='constrained')
for r, a in enumerate(C.ARMS):
    am = (arm == a).values
    for c_, name in enumerate(PROGRAMMES):
        ax = axes[r, c_]
        sel = am & (calls['programme'] == name).values
        C.scatter_umap(ax, xy[~am], C.GREY_BG, 0.3)
        C.scatter_umap(ax, xy[am & ~sel], '#bdbdbd', 0.35)
        C.scatter_umap(ax, xy[sel], C.PROG_COLORS[name], 0.9)
        C.scatter_umap(ax, xy, 'none', 0)
        nd = int((calls.loc[sel, 'donor'].value_counts() >= MIN_PER_DONOR).sum())
        ax.set_title(f'{a} — {name} ({PROGRAMMES[name]})\n{int(sel.sum()):,} cells, {nd} donor(s) ≥ {MIN_PER_DONOR}')
C.save(fig, f'{PLT}/programme_calls_umap_split.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/04_programme_calls/programme_calls_umap.pdf   183 x 78 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/04_programme_calls/programme_calls_umap_split.pdf   183 x 95 mm


## 5 · Sanity check against the harmonised annotation

Not a validation, just a look: the percentage of each population that lands in each programme.
Inflammatory calls should concentrate in Fb-interm, TGFb in Fb-TGFB+, adipose in Fb-adipo.

In [7]:
for a in C.ARMS:
    sub = calls[calls['arm'] == a]
    pct = (pd.crosstab(sub['population'], sub['programme'], normalize='index') * 100).reindex(columns=ORDER).fillna(0)
    pct = pct.loc[[p for p in C.POP_ORDER if p in pct.index]]
    pct.to_csv(f'{RES}/calls_by_population_{a}.csv')
    print(f'--- {C.ARM_NAMES[a]}: % of each population assigned to each programme')
    print(pct.round(1).to_string(), '\n')

--- scRNA-seq (cells): % of each population assigned to each programme
programme   TGFb  Inflammatory  Inflammatory_stress  Adipose_mature  unassigned
population                                                                     
Fb-interlo   5.8           1.4                 21.5             1.2        70.0
Fb-perilo    1.3           3.1                  0.3             4.5        90.7
Fb-med       1.0           1.9                  0.0             1.0        96.1
Fb-interm    7.8          60.1                 10.2             1.6        20.3
Fb-TGFB+    72.4           4.1                  1.1             0.5        21.9
Fb-adipo     0.9          22.5                  1.0            35.2        40.4 

--- snRNA-seq (multiome nuclei): % of each population assigned to each programme
programme   TGFb  Inflammatory  Inflammatory_stress  Adipose_mature  unassigned
population                                                                     
Fb-interlo   7.2           3.3                

## 6 · Export

One parquet with every cell's call and the scores behind it, the nuclei barcodes per programme
for the accessibility analysis, and a light `.h5ad` carrying the call in `.obs` next to the UMAP.
The calls are also checked against the table the accessibility analysis was started from.

In [8]:
out = f'{C.FINAL_DATA}/programme_calls_S1_S7_S12_S15.parquet'
calls.to_parquet(out)
pd.Series({'MIN_Z': MIN_Z, 'MIN_SHARE': MIN_SHARE, 'MIN_PER_DONOR': MIN_PER_DONOR, 'programmes': str(PROGRAMMES),
           'zscore': 'per arm, all donors pooled', 'assignment': 'exclusive: argmax of z among qualifying programmes'}
          ).to_csv(f'{RES}/programme_calls_params.csv')
print(f'-> {out}   {calls.shape}')

for name in PROGRAMMES:
    ids = calls.index[(calls['programme'] == name) & (calls['arm'] == 'nuclei')]
    pd.Series(ids, name='barcode').to_csv(f'{RES}/nuclei_barcodes_{name}.csv', index=False)
    print(f'  nuclei barcodes {name:20s} {len(ids):5,} -> {RES}/nuclei_barcodes_{name}.csv')

light.obs['programme'] = pd.Categorical(calls['programme'].reindex(light.obs_names), categories=ORDER)
light.obs['programme_z'] = calls['programme_z'].reindex(light.obs_names).values
light.write_h5ad(f'{C.FINAL_DATA}/fb_light_programme_calls.h5ad')
print(f'-> {C.FINAL_DATA}/fb_light_programme_calls.h5ad')

ref = f'{C.INMF_DIR}/programme_calls_k16_lam5_simple_z_0.75.parquet'
if os.path.exists(ref):
    prev = pd.read_parquet(ref)['programme'].reindex(calls.index)
    same = (prev.astype(str) == calls['programme'].astype(str))
    print(f'agreement with the archived z_0.75 calls: {same.mean():.4%} of cells ({int((~same).sum())} differ)')
    assert same.all(), 'the calls differ from the archived table'
display(calls.head())

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/programme_calls_S1_S7_S12_S15.parquet   (85116, 14)
  nuclei barcodes TGFb                   935 -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/final/04_programme_calls/nuclei_barcodes_TGFb.csv


  nuclei barcodes Inflammatory         1,810 -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/final/04_programme_calls/nuclei_barcodes_Inflammatory.csv
  nuclei barcodes Inflammatory_stress  1,547 -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/final/04_programme_calls/nuclei_barcodes_Inflammatory_stress.csv
  nuclei barcodes Adipose_mature       1,198 -> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/results/final/04_programme_calls/nuclei_barcodes_Adipose_mature.csv


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/fb_light_programme_calls.h5ad
agreement with the archived z_0.75 calls: 100.0000% of cells (0 differ)


,programme,programme_z,n_qualified,arm,donor,population,z_TGFb,share_TGFb,z_Inflammatory,share_Inflammatory,z_Inflammatory_stress,share_Inflammatory_stress,z_Adipose_mature,share_Adipose_mature
TA13072046-ACACCCTTCAGCACAT,unassigned,NaN,0,cells,A67,Fb-TGFB+,0.947,0.0969,-0.344,0.0615,-0.466,0.000,-0.309,0.0000
TA13072046-GCATGTACATGTAGTC,unassigned,NaN,0,cells,A67,Fb-TGFB+,0.573,0.0699,-0.426,0.0463,-0.466,0.000,-0.181,0.0091
TA13072046-CCGGTAGTCAGCTTAG,unassigned,NaN,0,cells,A67,Fb-TGFB+,-0.187,0.0148,-0.631,0.0084,-0.466,0.000,-0.309,0.0000
TA13072046-GGCCGATCAGTCGTGC,unassigned,NaN,0,cells,A67,Fb-TGFB+,0.269,0.0478,-0.670,0.0012,-0.410,0.007,-0.306,0.0003
TA13072046-CGATCGGGTACGCTGC,unassigned,NaN,0,cells,A67,Fb-TGFB+,0.034,0.0308,-0.382,0.0546,-0.466,0.000,-0.138,0.0121


## Files written

| file | contents |
|---|---|
| `data/nmf_joint/final/programme_calls_S1_S7_S12_S15.parquet` | barcode × (programme, z, n_qualified, per-programme z and share, arm, donor, population) |
| `data/nmf_joint/final/fb_light_programme_calls.h5ad` | the call in `.obs`, with the UMAP |
| `results/final/04_programme_calls/nuclei_barcodes_<programme>.csv` | the multiome nuclei of each programme |
| `results/final/04_programme_calls/calls_per_arm.csv`, `nuclei_per_programme_per_donor.csv`, `donor_support_nuclei.csv`, `threshold_sensitivity_nuclei.csv`, `calls_by_population_*.csv`, `programme_calls_params.csv` | tables |
| `plots/final/04_programme_calls/*.pdf` | donor-support heatmap, UMAPs |